# SpiderNet transfer-data preprocessing tutorial (AgingBrain Sagittal + Hippocampus)

This notebook prepares processed SpiderNet bundles for the **sagittal** and **hippocampus** transfer datasets used by `AgingBrain_transfer_Sagittal_Hippocampus_analysis_updated`.

It follows the unified `dataloading_unified` workflow, while preserving the important dataset-specific behavior from the original loaders:
- **Sagittal** keeps age-based batching, `center_x`/`center_y` spatial coordinates, and the same HVG-based preprocessing logic.
- **Hippocampus** reuses the **training gene list**, reads counts from `layers["raw_count"]`, and harmonizes hippocampus-specific cell-type labels before building the processed bundle.

In [ ]:
from workflow_paths import DATA_DIR, RESULTS_ROOT, input_path, output_path
from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

sys.path.append(str(Path.cwd()))

from SpiderNet.dataloading_unified import (
    prepare_processed_bundle_unified,
    preview_lr_corr_distribution,
)
from SpiderNet.utils import (
    get_default_cellchat_db,
    get_default_scseqcomm_db,
)


## Step 1. Basic user inputs

Edit these paths and preprocessing settings first.

In [ ]:
TRAIN_PROCESSED_DATA_DIR = (RESULTS_ROOT / 'ProcessedData')

SAGITTAL_DATA_ROOT = (DATA_DIR / 'AgingBrain_Sagittal')
SAGITTAL_OUTPUT_DIR = (RESULTS_ROOT / 'AgingBrain_Sagittal/ProcessedData')

HIPPOCAMPUS_DATA_ROOT = (DATA_DIR / 'AgingBrain_Hippocampus')
HIPPOCAMPUS_OUTPUT_DIR = (RESULTS_ROOT / 'AgingBrain_Hippocampus/ProcessedData')

SPECIES = "mouse"

# Keep these aligned with the transfer-analysis notebook unless you have a strong reason to change them.
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 5

# The original sagittal and hippocampus loaders both filtered LR pairs using a fixed correlation cutoff of 0.3.
# You can inspect the preview plots below before changing these values.
SAGITTAL_LR_CORR_THRESHOLD = 0.3
HIPPOCAMPUS_LR_CORR_THRESHOLD = 0.3

SHOW_PREVIEW_PLOTS = True


## Step 2. Dataset-specific preprocessing rules

This section encodes the sagittal- and hippocampus-specific behavior that used to live in `dataloading_AgingMousebrain` and `dataloading_AgingMouseHippocampus`.

In [ ]:
CELLCHAT_DB = get_default_cellchat_db(species=SPECIES)
SCSEQCOMM_DB = get_default_scseqcomm_db(species=SPECIES)

HIPPOCAMPUS_CELLTYPE_MAPPING = {
    "Astro-1": "Astrocyte",
    "Astro-2": "Astrocyte",
    "NPC-DG": "NSC",
    "Neuron-CA1": "Neuron-Excitatory",
    "Neuron-CA2&3": "Neuron-Excitatory",
    "OL-WM": "Oligodendrocyte",
}


def load_training_gene_list_path() -> Path:
    gene_list_path = TRAIN_PROCESSED_DATA_DIR / "genenames_train.pkl"
    if not input_path(gene_list_path).exists():
        raise FileNotFoundError(
            f"Training gene list was not found: {gene_list_path}. Run the training-data preprocessing notebook first."
        )
    return gene_list_path


def load_sagittal_gene_list_path(require_exists: bool = False) -> Path:
    gene_list_path = SAGITTAL_OUTPUT_DIR / "genenames_train.pkl"
    if require_exists and not input_path(gene_list_path).exists():
        raise FileNotFoundError(
            f"Sagittal gene list was not found: {gene_list_path}. Run the sagittal bundle cell first."
        )
    return gene_list_path


def extract_age_from_filename(file_name: str) -> float:
    stem = Path(file_name).stem
    if "_age" not in stem:
        raise ValueError(
            f"Cannot parse age from file name: {file_name}. Expected a pattern like '*_age12.h5ad'."
        )
    return float(stem.split("_age", 1)[1])


def sagittal_file_sort_key(file_name: str):
    return extract_age_from_filename(file_name)


def hippocampus_per_file_hook(adata: sc.AnnData, context: dict) -> sc.AnnData:
    adata = adata.copy()
    file_stem = Path(context["file_name"]).stem

    if "celltype" not in adata.obs.columns:
        if "cell_type" not in adata.obs.columns:
            raise KeyError(
                "Expected either 'celltype' or 'cell_type' in hippocampus AnnData.obs."
            )
        adata.obs["celltype"] = (
            adata.obs["cell_type"]
            .map(HIPPOCAMPUS_CELLTYPE_MAPPING)
            .fillna(adata.obs["cell_type"])
        )

    adata.obs["sample_name_full"] = file_stem
    return adata


def build_sagittal_config() -> dict:
    return {
        "data_path_main": SAGITTAL_DATA_ROOT,
        "output_dir": SAGITTAL_OUTPUT_DIR,
        "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
        "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,
        "sample_col": "age",
        "cell_class_col": "celltype",
        "pyg_obs_fields": {
            "age": "age",
            "batch": "age",
            "region": "region",
            "subregion": "subregion",
        },
        "file_sort_key": sagittal_file_sort_key,
        "expression_source": {"kind": "X", "name": None},
        "spatial_source": {"kind": "obs", "cols": ["center_x", "center_y"]},
        "normalize_strategy": "always",
        "n_hvg": N_HVG,
        "n_hvg_lr": N_HVG_LR,
        "num_neighbors": NUM_NEIGHBORS,
        "lr_corr_threshold": SAGITTAL_LR_CORR_THRESHOLD,
    }


def build_hippocampus_config() -> dict:
    sagittal_gene_list_path = load_sagittal_gene_list_path(require_exists=True)
    return {
        "data_path_main": HIPPOCAMPUS_DATA_ROOT,
        "output_dir": HIPPOCAMPUS_OUTPUT_DIR,
        "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
        "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,
        "sample_col": "sample_name_full",
        "sample_name_obs_col": "sample_name_full",
        "sample_attr_obs_col": "sample_name_full",
        "sample_sort": "lexicographic",
        "cell_class_col": "celltype",
        "per_file_hook": hippocampus_per_file_hook,
        "expression_source": {"kind": "layer", "name": "raw_count"},
        "spatial_source": {"kind": "obsm", "key": "spatial"},
        "gene_list_path": str(sagittal_gene_list_path),
        "fill_missing_requested_genes": True,
        "requested_gene_fill_value": 0.0,
        "apply_hvg_selection": False,
        "normalize_strategy": "always",
        "n_hvg": N_HVG,
        "n_hvg_lr": N_HVG_LR,
        "num_neighbors": NUM_NEIGHBORS,
        "lr_corr_threshold": HIPPOCAMPUS_LR_CORR_THRESHOLD,
    }


## Step 3. Quick path check

In [ ]:
rows = [
    {"item": "TRAIN_PROCESSED_DATA_DIR", "value": str(TRAIN_PROCESSED_DATA_DIR), "exists": input_path(TRAIN_PROCESSED_DATA_DIR).exists()},
    {"item": "SAGITTAL_DATA_ROOT", "value": str(SAGITTAL_DATA_ROOT), "exists": input_path(SAGITTAL_DATA_ROOT).exists()},
    {"item": "SAGITTAL_OUTPUT_DIR", "value": str(SAGITTAL_OUTPUT_DIR), "exists": input_path(SAGITTAL_OUTPUT_DIR).exists()},
    {"item": "Sagittal gene list (after Step 5)", "value": str(load_sagittal_gene_list_path()), "exists": input_path(load_sagittal_gene_list_path()).exists()},
    {"item": "HIPPOCAMPUS_DATA_ROOT", "value": str(HIPPOCAMPUS_DATA_ROOT), "exists": input_path(HIPPOCAMPUS_DATA_ROOT).exists()},
    {"item": "HIPPOCAMPUS_OUTPUT_DIR", "value": str(HIPPOCAMPUS_OUTPUT_DIR), "exists": input_path(HIPPOCAMPUS_OUTPUT_DIR).exists()},
    {"item": "CellChat DB", "value": str(CELLCHAT_DB), "exists": input_path(Path(CELLCHAT_DB)).exists()},
    {"item": "scSeqComm DB", "value": str(SCSEQCOMM_DB), "exists": input_path(Path(SCSEQCOMM_DB)).exists()},
    {"item": "Training gene list (reference only)", "value": str(load_training_gene_list_path()), "exists": input_path(load_training_gene_list_path()).exists()},
]
display(pd.DataFrame(rows))

## Step 4. Optional preview: sagittal LR-correlation distribution

This preview stops before the final saving step. It is useful if you want to adjust `SAGITTAL_LR_CORR_THRESHOLD`.

In [ ]:
sagittal_preview = preview_lr_corr_distribution(
    build_sagittal_config(),
    preview_output_dir=SAGITTAL_OUTPUT_DIR / "_lr_corr_preview",
    show_plot=SHOW_PREVIEW_PLOTS,
)

sagittal_preview["summary"]


## Step 5. Build the sagittal processed bundle

In [ ]:
sagittal_bundle = prepare_processed_bundle_unified(build_sagittal_config())

print("Sagittal output dir:", sagittal_bundle["output_dir"])
print("Sagittal batches:", len(sagittal_bundle["batch_cell_unique"]))
print("Sagittal total cells:", sagittal_bundle["adata"].n_obs)
print("Sagittal training genes:", len(sagittal_bundle["genenames_train"]))
print("Sagittal training LR pairs:", len(sagittal_bundle["LR_list"]))


## Step 6. Optional preview: hippocampus LR-correlation distribution

This step uses the **Sagittal processed gene list** generated in Step 5, so run the sagittal bundle cell first.

In [ ]:
print("Using sagittal gene list:", load_sagittal_gene_list_path(require_exists=True))

hippocampus_preview = preview_lr_corr_distribution(
    build_hippocampus_config(),
    preview_output_dir=HIPPOCAMPUS_OUTPUT_DIR / "_lr_corr_preview",
    show_plot=SHOW_PREVIEW_PLOTS,
)

hippocampus_preview["summary"]

## Step 7. Build the hippocampus processed bundle

The hippocampus bundle is aligned to the Sagittal training-gene space before LR construction.

In [ ]:
print("Using sagittal gene list:", load_sagittal_gene_list_path(require_exists=True))

hippocampus_bundle = prepare_processed_bundle_unified(build_hippocampus_config())

print("Hippocampus output dir:", hippocampus_bundle["output_dir"])
print("Hippocampus batches:", len(hippocampus_bundle["batch_cell_unique"]))
print("Hippocampus total cells:", hippocampus_bundle["adata"].n_obs)
print("Hippocampus training genes:", len(hippocampus_bundle["genenames_train"]))
print("Hippocampus training LR pairs:", len(hippocampus_bundle["LR_list"]))

## Step 8. Final output summary

These are the processed-data directories that the updated transfer-analysis notebook expects.

In [ ]:
summary_df = pd.DataFrame([
    {
        "dataset": "Sagittal",
        "processed_dir": str(SAGITTAL_OUTPUT_DIR),
        "exists": input_path(SAGITTAL_OUTPUT_DIR).exists(),
        "bundle_summary": str(SAGITTAL_OUTPUT_DIR / "bundle_summary.json"),
        "n_cells": sagittal_bundle["adata"].n_obs,
        "n_genes_train": len(sagittal_bundle["genenames_train"]),
        "n_lr_pairs_train": len(sagittal_bundle["LR_list"]),
    },
    {
        "dataset": "Hippocampus",
        "processed_dir": str(HIPPOCAMPUS_OUTPUT_DIR),
        "exists": input_path(HIPPOCAMPUS_OUTPUT_DIR).exists(),
        "bundle_summary": str(HIPPOCAMPUS_OUTPUT_DIR / "bundle_summary.json"),
        "n_cells": hippocampus_bundle["adata"].n_obs,
        "n_genes_train": len(hippocampus_bundle["genenames_train"]),
        "n_lr_pairs_train": len(hippocampus_bundle["LR_list"]),
    },
])

display(summary_df)
